In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))

In [ ]:
from jubilee_controller import JubileeMotionController
import matplotlib.pyplot as plt
import time
import numpy as np 
from pynput import keyboard
import cv2
from datetime import datetime
from camera_controller import Camera
from email_sender import send_email
from image_processing import detect_circle


In [6]:
jubilee = JubileeMotionController()

In [ ]:
jubilee.reset()

In [7]:
jubilee.home_all(mesh_mode_z=False)
jubilee.protect_tools(True)
jubilee.move_xyz_absolute(z=50)

In [ ]:
import cv2
import numpy as np


class Camera:
    """ Classe para controle de uma ferramenta de câmera em um sistema Jubilee ou similar.

    Esta classe encapsula os comandos necessários para instalar, desinstalar
    e capturar imagens usando uma câmera acoplada à máquina. A movimentação
    é feita através da interface da máquina, enquanto a captura de imagens
    utiliza a biblioteca OpenCV.

    Atributos da classe
    ----------
    installed : bool
        Indica se a câmera está instalada no sistema.
    machine : objeto
        Interface da máquina controlada (deve implementar métodos como `move_xyz_absolute` e `gcode`).
    move_velocity : int
        Velocidade padrão para movimentação do cabeçote durante instalação/desinstalação."""

    def __init__(self, machine,parking_position_xy=(297,20),move_velocity = 3000):
        """
        Inicializa a ferramenta da câmera.

        Parâmetros
        ----------
        machine : Instância da Jubilee.
        """
        self.name = "Câmera"
        self.parking_position_x,self.parking_position_y = parking_position_xy
        self.installed = False
        self.machine = machine
        self.move_velocity = move_velocity

    def install(self):
        """
        Instala a ferramenta da câmera na máquina.

        Este método move o cabeçote para as coordenadas específicas
        necessárias para acoplar a câmera ao sistema Jubilee. As
        Coodernadas foram descobertas empíricamente e podem ser 
        alteradas se necessário.
        """
        if self.machine.tool == None:
            self.machine.protect_tools(on=False)

            self.machine.move_xyz_absolute(y=90, velocity=self.move_velocity)
            self.machine.move_xyz_absolute(x=self.parking_position_x, velocity=self.move_velocity)
            self.machine.gcode("G91 G1 U10 F600 G90")  
            self.machine.gcode("G91 G1 H1 U300 F3000 G90")  
            self.machine.move_xyz_absolute(y=self.parking_position_y, velocity=self.move_velocity)
            self.machine.gcode("G92 U20")  
            self.machine.gcode("G91 G1 U-10 F600 G90")  
            self.machine.gcode("G91 G1 H1 U-300 F3000 G90")
            self.machine.gcode("G92 U0") 
            self.machine.move_xyz_absolute(y=70, velocity=self.move_velocity)
            self.machine.move_xyz_absolute(x=50, y=120, velocity=self.move_velocity)

            if self.machine.mode_protect_tools:
                self.machine.protect_tools(on=True,min_xy=[50,90])
                self.machine.gcode("M208 Z40:320")
            
            self.machine.tool = self.name
        
        else: 
            print('Desinstale a última ferramenta')

        


    def uninstall(self):
        """
        Remove a ferramenta da câmera da máquina.

        Este método move o cabeçote para as coordenadas específicas
        necessárias para desacoplar a câmera do sistema Jubilee.
        """
        if  self.machine.tool == self.name:
            self.machine.protect_tools(on=False)

            self.machine.move_xyz_absolute(y=90, velocity=self.move_velocity)
            self.machine.move_xyz_absolute(x=self.parking_position_x, velocity=self.move_velocity)
            self.machine.move_xyz_absolute(y=self.parking_position_y, velocity=self.move_velocity)
            self.machine.gcode("G91 G1 U10 F600 G90")  
            self.machine.gcode("G91 G1 H1 U300 F3000 G90")  
            self.machine.move_xyz_absolute(y=70, velocity=self.move_velocity)
            self.machine.move_xyz_absolute(x=50, y=120, velocity=self.move_velocity)
            self.machine.gcode("G92 U20")  
            self.machine.gcode("G91 G1 U-10 F600 G90")  
            self.machine.gcode("G91 G1 H1 U-300 F3000 G90")
            self.machine.gcode("G92 U0") 

            if self.machine.mode_protect_tools:
                self.machine.protect_tools(on=True)
            
            self.machine.tool = None
                
    def photo(self, filename='captura.jpg', video_index=0, focus_value=None):
        """
        Captura uma imagem usando a câmera conectada.

        Parâmetros
        ----------
        filename : str
            Nome do arquivo de saída.
        video_index : int
            Índice da câmera.
        focus_value : int ou None
            Valor do foco manual. Se None, mantém a configuração atual.
        """

        cap = cv2.VideoCapture(video_index)

        if not cap.isOpened():
            return

        cap.set(cv2.CAP_PROP_AUTOFOCUS, 0)

        if focus_value is not None:
            cap.set(cv2.CAP_PROP_FOCUS, focus_value)

        for _ in range(20):
            ret, frame = cap.read()
            if not ret:
                cap.release()
                return

        ret, frame = cap.read()

        if ret:
            cv2.imwrite(filename, frame)

        cap.release()

    
    def photo_autofocus(self, altura_z, filename='captura.jpg', video_index=0):
        """
        Captura uma imagem com foco automático usando a câmera conectada.

        Parâmetros
        ----------
        altura_z : float
            Altura do eixo Z da Jubilee.
        filename : str
            Nome do arquivo de saída.
        video_index : int
            Índice da câmera.
        """

        cap = cv2.VideoCapture(video_index)

        if not cap.isOpened():
            return

        auto_focus_value = 17.890934 + 265.025880 * np.exp(-0.033187 * altura_z)

        cap.set(cv2.CAP_PROP_FOCUS, auto_focus_value)

        for _ in range(20):
            ret, frame = cap.read()
            if not ret:
                cap.release()
                return

        ret, frame = cap.read()

        if ret:
            cv2.imwrite(filename, frame)

        cap.release()

In [ ]:
camera = Camera(jubilee)

In [ ]:
camera.install()
jubilee.move_xyz_absolute(x=230,y=20,z=50,velocity=3000)